# 05 Run Report And Go/No-Go
Stage-Metriken aggregieren und Gate-Entscheidung schreiben.

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

RUN_ID = "replace_with_run_id_from_00"
import json
import pandas as pd


ROOT: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation


In [2]:
from src.common.config import load_notebook_context, build_run_dirs
from src.common.io_schema import read_parquet, validate_columns, MENTION_REQUIRED_COLUMNS
from src.common.run_report import evaluate_go_no_go, write_go_no_go_report

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]

RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
metrics_dir = RUN_DIRS["metrics"]
subset_dir = RUN_DIRS["subset_cache"]
cluster_dir = RUN_DIRS["clusters"]
manifest_dir = RUN_DIRS["subset_manifests"]

stage_metrics = {}


In [3]:
train_manifest = {}
train_manifest_path = metrics_dir / "03_train_manifest.json"
if train_manifest_path.exists():
    with train_manifest_path.open("r", encoding="utf-8") as f:
        train_manifest = json.load(f)

lspo_mentions = read_parquet(subset_dir / f"lspo_mentions_{RUN_STAGE}.parquet")
ads_mentions = read_parquet(subset_dir / f"ads_mentions_{RUN_STAGE}.parquet")
clusters_path = cluster_dir / f"ads_clusters_{RUN_STAGE}.parquet"
clusters = read_parquet(clusters_path) if clusters_path.exists() else pd.DataFrame(columns=["mention_id", "block_key", "author_uid"])

schema_valid = True
try:
    validate_columns(lspo_mentions, MENTION_REQUIRED_COLUMNS, "lspo_mentions")
    validate_columns(ads_mentions, MENTION_REQUIRED_COLUMNS, "ads_mentions")
except Exception:
    schema_valid = False

determinism_valid = bool((manifest_dir / f"{RUN_ID}_lspo_{RUN_STAGE}_manifest.parquet").exists())
uid_uniqueness_valid = clusters.groupby("mention_id")["author_uid"].nunique().max() == 1 if len(clusters) else False
lspo_pairwise_f1 = float(train_manifest["best_val_f1"]) if train_manifest.get("best_val_f1") is not None else None

stage_metrics.update({
    "run_id": RUN_ID,
    "stage": RUN_STAGE,
    "schema_valid": schema_valid,
    "determinism_valid": determinism_valid,
    "uid_uniqueness_valid": bool(uid_uniqueness_valid),
    "lspo_pairwise_f1": lspo_pairwise_f1,
    "counts": {
        "lspo_mentions": int(len(lspo_mentions)),
        "ads_mentions": int(len(ads_mentions)),
        "ads_clusters": int(len(clusters)),
    },
})
stage_metrics


{'run_id': 'replace_with_run_id_from_00',
 'stage': 'smoke',
 'schema_valid': True,
 'determinism_valid': True,
 'uid_uniqueness_valid': True,
 'lspo_pairwise_f1': 0.9850746268656716,
 'counts': {'lspo_mentions': 1000, 'ads_mentions': 1000, 'ads_clusters': 1000}}

In [4]:
go = evaluate_go_no_go(stage_metrics)
report_path = metrics_dir / f"05_go_no_go_{RUN_STAGE}.json"
summary_path = metrics_dir / f"05_stage_metrics_{RUN_STAGE}.json"

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(stage_metrics, f, indent=2)
write_go_no_go_report(go, report_path)

print("GO:" if go["go"] else "NO-GO:", go["go"])
display(pd.DataFrame(go["checks"]))
print("stage summary:", summary_path)
print("go/no-go:", report_path)


GO: True


,name,passed,detail
0,schema_valid,True,All required schemas validated
1,determinism_valid,True,Determinism checks passed
2,uid_uniqueness,True,Each mention_id mapped to one author_uid
3,lspo_pairwise_f1_sanity,True,Observed F1=0.9851


stage summary: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\artifacts\metrics\replace_with_run_id_from_00\05_stage_metrics_smoke.json
go/no-go: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\artifacts\metrics\replace_with_run_id_from_00\05_go_no_go_smoke.json
